# Notebook 38 · Find new winter JPCZ events and update the shared catalog

**This is the update entry point for Notebooks 35, 36, and 37.** Start here to search
for new events; you do not need to rerun the historical catalog or the old map batches.
The detector is embedded, so an old clone or the repository's `main` settings cannot change it.

The established workflow is [Notebook 04: detector](https://colab.research.google.com/github/angelicasophyaramirez-blip/JPCZcatalogcolab/blob/codex/notebook36-weathernext2/notebooks/04_ndjf_catalog.ipynb)
then [Notebook 06: 12-hour fragment merge](https://colab.research.google.com/github/angelicasophyaramirez-blip/JPCZcatalogcolab/blob/codex/notebook36-weathernext2/notebooks/06_gap_merge_catalog.ipynb).
**Notebook 34 is a separate historical-baseline experiment**, not the rolling extension.

This version uses hourly **925-hPa ERA5 winds → MetPy grid divergence → cosine-latitude
polygon mean → trailing 12-hour mean → D12 strictly below a frozen month-specific
mean − 2 population standard deviations → 12-hour fragment merge**. It preserves Notebook
04's 11-hour warmup at each month boundary. These are this project's Shinoda-style ERA5
implementation choices. Vortex labels are a separate review in Notebook 35, not a detection filter.

Your actual Drive CSV supplies the count (whether it is 275 or another number). Old events
and threshold files are backed up before the first update. New thresholds are never fitted to
the new winter. Scans checkpoint compact scalar series; a failed month cannot promote a partial catalog.


In [ ]:
# @title 1 · Install dependencies
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'numpy>=1.26,<2.4', 'pandas>=2.2,<3', 'xarray>=2026.4,<2027',
    'zarr>=3.1,<4', 'dask[array]>=2025.1,<2027', 'gcsfs>=2025.3,<2027',
    'matplotlib>=3.9,<3.11', 'metpy>=1.6,<2', 'h5netcdf', 'scipy'], check=True)


In [ ]:
# @title 2 · Mount Drive and set the update window
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import pandas as pd
from IPython.display import display
DRIVE_ROOT = Path('/content/drive/MyDrive/JPCZcatalog_outputs')
UPDATE_START_UTC = ''  # Optional first-run month boundary; otherwise infer from a complete saved month.
UPDATE_THROUGH_UTC = ''  # Inclusive UTC hour; blank means now. Example: '2026-11-19T23:00:00Z'.
ALLOW_ERA5T = True  # Preliminary reference, explicitly marked and refreshed on rerun.


In [ ]:
# @title 3 · Load the embedded detector (no Git branch checkout required)
import sys, types
sources = {'config': '"""Shared project configuration for Colab notebooks and helper modules."""\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom typing import Final\n\n\n@dataclass(frozen=True)\nclass BoundingBox:\n    lon_min: float\n    lon_max: float\n    lat_min: float\n    lat_max: float\n\n\n@dataclass(frozen=True)\nclass GeographicPoint:\n    name: str\n    latitude: float\n    longitude: float\n\n\nARCO_ERA5_ZARR_STORE: Final[str] = (\n    "gs://gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3"\n)\n\nWORKING_DOMAIN = BoundingBox(\n    lon_min=120.0,\n    lon_max=150.0,\n    lat_min=30.0,\n    lat_max=50.0,\n)\n\nEXTENDED_DOMAIN = BoundingBox(\n    lon_min=120.0,\n    lon_max=160.0,\n    lat_min=30.0,\n    lat_max=50.0,\n)\n\n# Tighter domain for objective subtype characterization and clustering.\n# This still covers the JPCZ polygon plus the coastal, Pacific, Hokkaido,\n# Sea of Japan, and frontal characterization regions, while reducing\n# Colab memory pressure compared with the larger EXTENDED_DOMAIN.\nOBJECTIVE_SUBTYPE_DOMAIN = BoundingBox(\n    lon_min=129.0,\n    lon_max=151.0,\n    lat_min=33.0,\n    lat_max=47.0,\n)\n\nVORTICITY_BOX = BoundingBox(\n    lon_min=127.0,\n    lon_max=140.0,\n    lat_min=37.0,\n    lat_max=45.5,\n)\n\n# First-pass characterization regions for objective subtype analysis.\n# These do not affect event detection; they only describe where convergence\n# and forcing are strongest after the Shinoda-style catalog has been built.\nCOASTAL_JAPAN_BOX = BoundingBox(\n    lon_min=131.0,\n    lon_max=140.0,\n    lat_min=34.0,\n    lat_max=40.5,\n)\n\nPACIFIC_EAST_OF_JAPAN_BOX = BoundingBox(\n    lon_min=141.0,\n    lon_max=150.0,\n    lat_min=33.0,\n    lat_max=42.0,\n)\n\nHOKKAIDO_BOX = BoundingBox(\n    lon_min=139.0,\n    lon_max=146.5,\n    lat_min=41.0,\n    lat_max=46.5,\n)\n\nSEA_OF_JAPAN_BOX = BoundingBox(\n    lon_min=129.0,\n    lon_max=140.0,\n    lat_min=36.0,\n    lat_max=45.0,\n)\n\nHOKKAIDO_FRONT_BOX = BoundingBox(\n    lon_min=136.0,\n    lon_max=147.0,\n    lat_min=39.0,\n    lat_max=47.0,\n)\n\nPACIFIC_FRONT_BOX = BoundingBox(\n    lon_min=141.0,\n    lon_max=151.0,\n    lat_min=33.0,\n    lat_max=42.0,\n)\n\n# Terrain-sensitive low-level metrics can spuriously light up along the\n# Russian coast, so several notebooks mask those grid cells before\n# computing divergence and moisture-proxy summaries.\nRUSSIAN_COASTAL_EXCLUSION_BOXES = (\n    BoundingBox(lon_min=130.5, lon_max=135.5, lat_min=42.0, lat_max=45.0),\n    BoundingBox(lon_min=133.5, lon_max=139.5, lat_min=44.0, lat_max=47.25),\n)\n\n# First-pass digitization from Shinoda Figure 2. These values are intended\n# to be refined only if validation indicates a clear mismatch.\nJPCZ_POLYGON_VERTICES = (\n    (129.5, 41.0),\n    (136.0, 37.4),\n    (134.5, 35.8),\n    (128.8, 38.0),\n)\n\nSEOUL = GeographicPoint(name="Seoul", latitude=37.5665, longitude=126.9780)\nSAPPORO = GeographicPoint(name="Sapporo", latitude=43.0618, longitude=141.3545)\n\nDECEMBER_BENCHMARK_YEARS: Final[tuple[int, ...]] = tuple(range(2000, 2019))\nBASELINE_START_UTC: Final[str] = "2018-02-02"\nBASELINE_END_UTC: Final[str] = "2018-02-07T23:00:00"\nBASELINE_PEAK_DATE_UTC: Final[str] = "2018-02-03"\n\nEVENT_FIELD_UNITS = {\n    "divergence_925hpa": "s^-1",\n    "divergence_925hpa_display": "1e-5 s^-1",\n    "relative_vorticity_925hpa": "s^-1",\n    "relative_vorticity_925hpa_display": "1e-5 s^-1",\n    "geopotential_height_anomaly_850hpa": "gpm",\n    "temperature_gradient_850hpa_display": "K (100 km)^-1",\n    "seoul_minus_sapporo_slp": "hPa",\n}\n', 'masks': '"""Mask construction helpers for JPCZ polygons and vorticity boxes."""\n\nfrom __future__ import annotations\n\nfrom collections.abc import Sequence\n\nimport numpy as np\nimport xarray as xr\nfrom matplotlib.path import Path as MplPath\n\nfrom .config import BoundingBox\n\n\ndef _coord_values(coord: xr.DataArray | np.ndarray | Sequence[float]) -> np.ndarray:\n    values = getattr(coord, "values", coord)\n    return np.asarray(values)\n\n\ndef build_polygon_mask(\n    longitude: xr.DataArray | np.ndarray | Sequence[float],\n    latitude: xr.DataArray | np.ndarray | Sequence[float],\n    polygon_vertices: Sequence[tuple[float, float]],\n) -> xr.DataArray:\n    """Return a boolean mask for the polygon on a lat-lon grid."""\n    lon_values = _coord_values(longitude)\n    lat_values = _coord_values(latitude)\n    lon2d, lat2d = np.meshgrid(lon_values, lat_values)\n    polygon_path = MplPath(polygon_vertices)\n    mask = polygon_path.contains_points(\n        np.column_stack([lon2d.ravel(), lat2d.ravel()])\n    ).reshape(lat2d.shape)\n    return xr.DataArray(\n        mask,\n        coords={"latitude": lat_values, "longitude": lon_values},\n        dims=("latitude", "longitude"),\n        name="jpcz_polygon_mask",\n    )\n\n\ndef build_coslat_weights(\n    latitude: xr.DataArray | np.ndarray | Sequence[float],\n    longitude: xr.DataArray | np.ndarray | Sequence[float],\n    *,\n    mask: xr.DataArray | np.ndarray | None = None,\n) -> xr.DataArray:\n    """Return cosine-latitude area weights, optionally masked."""\n    lon_values = _coord_values(longitude)\n    lat_values = _coord_values(latitude)\n    lon2d, lat2d = np.meshgrid(lon_values, lat_values)\n    weights = np.cos(np.deg2rad(lat2d))\n\n    if mask is not None:\n        mask_values = np.asarray(getattr(mask, "values", mask), dtype=float)\n        weights = weights * mask_values\n\n    return xr.DataArray(\n        weights,\n        coords={"latitude": lat_values, "longitude": lon_values},\n        dims=("latitude", "longitude"),\n        name="area_weights",\n    )\n\n\ndef box_outline(box: BoundingBox) -> tuple[list[float], list[float]]:\n    """Return longitude and latitude vertices for plotting a bounding box."""\n    lons = [\n        box.lon_min,\n        box.lon_max,\n        box.lon_max,\n        box.lon_min,\n        box.lon_min,\n    ]\n    lats = [\n        box.lat_min,\n        box.lat_min,\n        box.lat_max,\n        box.lat_max,\n        box.lat_min,\n    ]\n    return lons, lats\n\n\ndef split_polygon_land_ocean_mask(\n    longitude: xr.DataArray,\n    latitude: xr.DataArray,\n    polygon_vertices: Sequence[tuple[float, float]],\n) -> tuple[xr.DataArray, xr.DataArray, xr.DataArray]:\n    """Split polygon cells into ocean and land subsets using Natural Earth."""\n    import regionmask\n\n    polygon_mask = build_polygon_mask(longitude, latitude, polygon_vertices)\n\n    defined = regionmask.defined_regions\n    if hasattr(defined, "natural_earth_v5_1_2"):\n        ne = defined.natural_earth_v5_1_2\n    elif hasattr(defined, "natural_earth_v5_0_0"):\n        ne = defined.natural_earth_v5_0_0\n    elif hasattr(defined, "natural_earth_v4_1_0"):\n        ne = defined.natural_earth_v4_1_0\n    else:\n        raise RuntimeError("No supported Natural Earth region set found in regionmask.")\n\n    land_regions = getattr(ne, "land_50", getattr(ne, "land_110"))\n    land_mask = land_regions.mask(longitude, latitude)\n    ocean_mask = np.isnan(land_mask.values)\n\n    polygon_ocean_mask = xr.DataArray(\n        polygon_mask.values & ocean_mask,\n        coords=polygon_mask.coords,\n        dims=polygon_mask.dims,\n        name="polygon_ocean_mask",\n    )\n    polygon_land_mask = xr.DataArray(\n        polygon_mask.values & (~ocean_mask),\n        coords=polygon_mask.coords,\n        dims=polygon_mask.dims,\n        name="polygon_land_mask",\n    )\n\n    return polygon_mask, polygon_ocean_mask, polygon_land_mask\n\n', 'detect': '"""JPCZ detection helpers based on Shinoda-style divergence thresholds."""\n\nfrom __future__ import annotations\n\nfrom collections.abc import Iterable, Sequence\nfrom dataclasses import dataclass\n\nimport numpy as np\nimport pandas as pd\nimport xarray as xr\nimport metpy.calc as mpcalc\nfrom metpy.units import units\n\nfrom .masks import build_coslat_weights, build_polygon_mask\n\n\n@dataclass(frozen=True)\nclass DetectionGeometry:\n    """Reusable grid geometry for repeated divergence calculations."""\n\n    dx: object\n    dy: object\n    polygon_mask: xr.DataArray\n    weights: xr.DataArray\n\n\ndef compute_grid_deltas(\n    longitude: xr.DataArray | np.ndarray,\n    latitude: xr.DataArray | np.ndarray,\n):\n    """Compute horizontal grid spacing on a lat-lon grid."""\n    lon_values = np.asarray(getattr(longitude, "values", longitude))\n    lat_values = np.asarray(getattr(latitude, "values", latitude))\n    return mpcalc.lat_lon_grid_deltas(lon_values, lat_values)\n\n\ndef prepare_detection_geometry(\n    longitude: xr.DataArray | np.ndarray,\n    latitude: xr.DataArray | np.ndarray,\n    polygon_vertices: Sequence[tuple[float, float]],\n) -> DetectionGeometry:\n    """Precompute reusable geometry for one lat-lon grid."""\n    polygon_mask = build_polygon_mask(longitude, latitude, polygon_vertices)\n    weights = build_coslat_weights(\n        latitude,\n        longitude,\n        mask=polygon_mask,\n    )\n    dx, dy = compute_grid_deltas(longitude, latitude)\n    return DetectionGeometry(\n        dx=dx,\n        dy=dy,\n        polygon_mask=polygon_mask,\n        weights=weights,\n    )\n\n\ndef compute_divergence_field(\n    ds_time: xr.Dataset,\n    *,\n    u_name: str = "u_component_of_wind",\n    v_name: str = "v_component_of_wind",\n    dx=None,\n    dy=None,\n) -> xr.DataArray:\n    """Compute 925 hPa divergence for one analysis time."""\n    snapshot = ds_time\n    if "time" in snapshot.dims:\n        if snapshot.sizes["time"] != 1:\n            raise ValueError("compute_divergence_field expects a single analysis time.")\n        snapshot = snapshot.isel(time=0, drop=True)\n\n    if dx is None or dy is None:\n        dx, dy = compute_grid_deltas(snapshot.longitude, snapshot.latitude)\n\n    u = snapshot[u_name].values * units("m/s")\n    v = snapshot[v_name].values * units("m/s")\n    div = mpcalc.divergence(u, v, dx=dx, dy=dy).m\n\n    return xr.DataArray(\n        div,\n        coords={\n            "latitude": snapshot.latitude,\n            "longitude": snapshot.longitude,\n        },\n        dims=("latitude", "longitude"),\n        name="divergence_925hpa",\n        attrs={"units": "s^-1", "display_units": "1e-5 s^-1"},\n    )\n\n\ndef compute_divergence_stack(\n    window_ds: xr.Dataset,\n    *,\n    u_name: str = "u_component_of_wind",\n    v_name: str = "v_component_of_wind",\n    dx=None,\n    dy=None,\n    progress_every: int | None = None,\n    progress_label: str = "Divergence",\n) -> xr.DataArray:\n    """Compute hourly 925 hPa divergence over a time window."""\n    if dx is None or dy is None:\n        dx, dy = compute_grid_deltas(window_ds.longitude, window_ds.latitude)\n\n    divergence_stack = []\n    for i in range(window_ds.sizes["time"]):\n        divergence_stack.append(\n            compute_divergence_field(\n                window_ds.isel(time=i),\n                u_name=u_name,\n                v_name=v_name,\n                dx=dx,\n                dy=dy,\n            ).values\n        )\n        if progress_every and (\n            (i + 1) % progress_every == 0 or (i + 1) == window_ds.sizes["time"]\n        ):\n            print(f"{progress_label}: divergence {i + 1}/{window_ds.sizes[\'time\']} hours")\n\n    return xr.DataArray(\n        np.stack(divergence_stack, axis=0),\n        coords={\n            "time": window_ds.time,\n            "latitude": window_ds.latitude,\n            "longitude": window_ds.longitude,\n        },\n        dims=("time", "latitude", "longitude"),\n        name="divergence_925hpa",\n        attrs={"units": "s^-1", "display_units": "1e-5 s^-1"},\n    )\n\n\ndef compute_rolling_mean(\n    series: xr.DataArray,\n    *,\n    window: int = 12,\n    min_periods: int | None = None,\n    name: str = "D_12h",\n) -> xr.DataArray:\n    """Compute a rolling-mean time series."""\n    if min_periods is None:\n        min_periods = window\n    rolled = series.rolling(time=window, min_periods=min_periods).mean().rename(name)\n    rolled.attrs["units"] = "s^-1"\n    rolled.attrs["display_units"] = "1e-5 s^-1"\n    return rolled\n\n\ndef compute_polygon_mean_divergence_series(\n    window_ds: xr.Dataset,\n    polygon_vertices: Sequence[tuple[float, float]] | None = None,\n    *,\n    geometry: DetectionGeometry | None = None,\n    progress_every: int | None = None,\n    progress_label: str = "Divergence",\n) -> tuple[xr.DataArray, xr.DataArray, xr.DataArray]:\n    """Compute hourly polygon-mean divergence and Shinoda-style 12-hour mean D."""\n    loaded = window_ds.load()\n    if geometry is None:\n        if polygon_vertices is None:\n            raise ValueError("Provide either polygon_vertices or a precomputed geometry.")\n        geometry = prepare_detection_geometry(\n            loaded.longitude,\n            loaded.latitude,\n            polygon_vertices,\n        )\n\n    div_925 = compute_divergence_stack(\n        loaded,\n        dx=geometry.dx,\n        dy=geometry.dy,\n        progress_every=progress_every,\n        progress_label=progress_label,\n    )\n    polygon_mask = geometry.polygon_mask\n    weights = geometry.weights\n\n    hourly = (\n        (div_925 * weights).sum(dim=("latitude", "longitude"))\n        / weights.sum(dim=("latitude", "longitude"))\n    ).rename("polygon_mean_divergence_925hpa")\n    hourly.attrs["units"] = "s^-1"\n    hourly.attrs["display_units"] = "1e-5 s^-1"\n\n    D = compute_rolling_mean(hourly, window=12, min_periods=12, name="D_12h")\n    return hourly, D, polygon_mask\n\n\ndef threshold_from_std(\n    D_series: xr.DataArray,\n    *,\n    n_std: float = 2.0,\n) -> tuple[float, float, float]:\n    """Return mean, std, and Shinoda-style anomaly threshold."""\n    valid = D_series.dropna("time")\n    D_mean = float(valid.mean().values)\n    D_std = float(valid.std().values)\n    threshold = D_mean - n_std * D_std\n    return D_mean, D_std, threshold\n\n\ndef detect_threshold_events(D_series: xr.DataArray, threshold: float) -> pd.DataFrame:\n    """Group consecutive threshold crossings into events."""\n    valid = D_series.dropna("time")\n    threshold_hits = (valid < threshold).to_series()\n\n    events: list[dict[str, object]] = []\n    in_event = False\n    current_times: list[pd.Timestamp] = []\n\n    for timestamp, hit in threshold_hits.items():\n        if hit and not in_event:\n            in_event = True\n            current_times = [timestamp]\n        elif hit and in_event:\n            current_times.append(timestamp)\n        elif (not hit) and in_event:\n            events.append(_build_event_record(valid, current_times))\n            in_event = False\n            current_times = []\n\n    if in_event:\n        events.append(_build_event_record(valid, current_times))\n\n    return pd.DataFrame(events)\n\n\ndef _build_event_record(\n    D_series: xr.DataArray,\n    current_times: Sequence[pd.Timestamp],\n) -> dict[str, object]:\n    event_slice = D_series.sel(time=current_times)\n    peak_time = pd.Timestamp(event_slice.idxmin("time").values)\n    peak_value = float(event_slice.min().values)\n    return {\n        "event_start": current_times[0],\n        "event_end": current_times[-1],\n        "event_peak": peak_time,\n        "event_peak_D_s-1": peak_value,\n        "event_peak_D_1e5_s-1": peak_value * 1e5,\n        "duration_hours": len(current_times),\n    }\n\n\ndef count_events_from_threshold(D_series: xr.DataArray, threshold: float) -> pd.DataFrame:\n    """Small convenience wrapper for threshold event detection."""\n    return detect_threshold_events(D_series, threshold)\n\n\ndef merge_nearby_events(\n    events_df: pd.DataFrame,\n    *,\n    max_gap_hours: float,\n) -> pd.DataFrame:\n    """Merge adjacent events separated by small threshold-free gaps.\n\n    The merged event keeps the strongest-peak row as the representative\n    environment/classification record, while broadening the event start/end\n    window to cover the full episode.\n    """\n    if events_df.empty:\n        return events_df.copy()\n\n    events = events_df.copy()\n    for column_name in ("event_start", "event_end", "event_peak"):\n        events[column_name] = pd.to_datetime(events[column_name])\n\n    events = events.sort_values("event_start").reset_index(drop=True)\n\n    clusters: list[list[int]] = [[0]]\n    for idx in range(1, len(events)):\n        previous_end = events.loc[clusters[-1][-1], "event_end"]\n        current_start = events.loc[idx, "event_start"]\n        gap_hours = (current_start - previous_end).total_seconds() / 3600.0\n        if gap_hours <= max_gap_hours:\n            clusters[-1].append(idx)\n        else:\n            clusters.append([idx])\n\n    merged_rows: list[pd.Series] = []\n    for cluster in clusters:\n        cluster_df = events.loc[cluster].copy()\n        peak_idx = cluster_df["event_peak_D_s-1"].astype(float).idxmin()\n        merged_row = cluster_df.loc[peak_idx].copy()\n\n        event_start = cluster_df["event_start"].iloc[0]\n        event_end = cluster_df["event_end"].iloc[-1]\n        internal_gaps = (\n            cluster_df["event_start"].iloc[1:].reset_index(drop=True)\n            - cluster_df["event_end"].iloc[:-1].reset_index(drop=True)\n        ).dt.total_seconds() / 3600.0\n\n        episode_span_hours = int((event_end - event_start).total_seconds() / 3600.0) + 1\n        threshold_hit_hours = int(pd.to_numeric(cluster_df["duration_hours"]).sum())\n\n        merged_row["event_start"] = event_start\n        merged_row["event_end"] = event_end\n        merged_row["duration_hours"] = episode_span_hours\n        merged_row["episode_span_hours"] = episode_span_hours\n        merged_row["threshold_hit_hours"] = threshold_hit_hours\n        merged_row["merged_subevents_count"] = int(len(cluster_df))\n        merged_row["total_internal_gap_hours"] = float(internal_gaps.sum()) if len(internal_gaps) else 0.0\n        merged_row["max_internal_gap_hours"] = float(internal_gaps.max()) if len(internal_gaps) else 0.0\n\n        merged_rows.append(merged_row)\n\n    merged_df = pd.DataFrame(merged_rows).sort_values("event_start").reset_index(drop=True)\n    return merged_df\n\n\ndef merge_gap_sensitivity(\n    events_df: pd.DataFrame,\n    *,\n    gap_hours_values: Iterable[float] = (6, 12, 24),\n) -> pd.DataFrame:\n    """Summarize how the event count changes under different merge gaps."""\n    rows = [\n        {\n            "gap_hours": 0.0,\n            "event_count": int(len(events_df)),\n            "events_merged": 0,\n        }\n    ]\n\n    baseline_count = len(events_df)\n    for gap_hours in gap_hours_values:\n        merged_df = merge_nearby_events(events_df, max_gap_hours=gap_hours)\n        merged_count = len(merged_df)\n        rows.append(\n            {\n                "gap_hours": float(gap_hours),\n                "event_count": int(merged_count),\n                "events_merged": int(baseline_count - merged_count),\n            }\n        )\n\n    return pd.DataFrame(rows)\n\n\ndef threshold_sensitivity(\n    D_series: xr.DataArray,\n    n_std_values: Iterable[float],\n) -> pd.DataFrame:\n    """Evaluate event-count sensitivity to the anomaly threshold."""\n    D_mean, D_std, _ = threshold_from_std(D_series, n_std=2.0)\n    rows = []\n    for n_std in n_std_values:\n        threshold = D_mean - n_std * D_std\n        events_df = count_events_from_threshold(D_series, threshold)\n        rows.append(\n            {\n                "threshold_label": f"mean - {n_std:.1f} std",\n                "n_std": n_std,\n                "threshold_s-1": threshold,\n                "threshold_1e5_s-1": threshold * 1e5,\n                "event_count": len(events_df),\n            }\n        )\n    return pd.DataFrame(rows)\n', 'era5': '"""Cloud-hosted ERA5 access helpers."""\n\nfrom __future__ import annotations\n\nimport calendar\nfrom collections.abc import Iterable, Sequence\n\nimport xarray as xr\n\nfrom .config import (\n    ARCO_ERA5_ZARR_STORE,\n    BASELINE_END_UTC,\n    BASELINE_START_UTC,\n    BoundingBox,\n    DECEMBER_BENCHMARK_YEARS,\n    WORKING_DOMAIN,\n)\n\nANALYSIS_VARIABLES: tuple[str, ...] = (\n    "u_component_of_wind",\n    "v_component_of_wind",\n    "mean_sea_level_pressure",\n)\n\n\ndef open_arco_era5(\n    store: str = ARCO_ERA5_ZARR_STORE,\n    *,\n    chunks: dict[str, int] | None = None,\n    storage_options: dict[str, str] | None = None,\n) -> xr.Dataset:\n    """Open the public ARCO ERA5 Zarr store with anonymous access."""\n    if storage_options is None:\n        storage_options = {"token": "anon"}\n    return xr.open_zarr(store, chunks=chunks, storage_options=storage_options)\n\n\ndef subset_era5_window(\n    ds: xr.Dataset,\n    start: str,\n    end: str,\n    *,\n    domain: BoundingBox = WORKING_DOMAIN,\n    variables: Sequence[str] = ANALYSIS_VARIABLES,\n    level: int = 925,\n) -> xr.Dataset:\n    """Subset ERA5 by time, domain, and optional level."""\n    subset = ds[list(variables)].sel(\n        time=slice(start, end),\n        longitude=slice(domain.lon_min, domain.lon_max),\n        latitude=slice(domain.lat_max, domain.lat_min),\n    )\n\n    if "level" in subset.dims or "level" in subset.coords:\n        subset = subset.sel(level=level)\n\n    return subset\n\n\ndef december_window(year: int) -> tuple[str, str]:\n    """Return the start/end slice for one December."""\n    return f"{year}-12-01", f"{year}-12-31T23:00:00"\n\n\ndef month_window(year: int, month: int) -> tuple[str, str]:\n    """Return the start/end slice for one calendar month."""\n    last_day = calendar.monthrange(year, month)[1]\n    return f"{year}-{month:02d}-01", f"{year}-{month:02d}-{last_day:02d}T23:00:00"\n\n\ndef iter_december_windows(\n    years: Iterable[int] = DECEMBER_BENCHMARK_YEARS,\n) -> Iterable[tuple[int, str, str]]:\n    """Yield year/start/end tuples for December benchmark windows."""\n    for year in years:\n        start, end = december_window(year)\n        yield year, start, end\n\n\ndef iter_month_windows(\n    years: Iterable[int],\n    months: Iterable[int],\n) -> Iterable[tuple[int, int, str, str]]:\n    """Yield calendar-year/month start/end tuples for month-based catalog runs."""\n    for year in years:\n        for month in months:\n            start, end = month_window(year, month)\n            yield year, month, start, end\n\n\ndef open_december_month(\n    ds: xr.Dataset,\n    year: int,\n    *,\n    domain: BoundingBox = WORKING_DOMAIN,\n    variables: Sequence[str] = ANALYSIS_VARIABLES,\n    level: int = 925,\n) -> xr.Dataset:\n    """Open one December analysis window."""\n    start, end = december_window(year)\n    return subset_era5_window(\n        ds,\n        start,\n        end,\n        domain=domain,\n        variables=variables,\n        level=level,\n    )\n\n\ndef open_month(\n    ds: xr.Dataset,\n    year: int,\n    month: int,\n    *,\n    domain: BoundingBox = WORKING_DOMAIN,\n    variables: Sequence[str] = ANALYSIS_VARIABLES,\n    level: int = 925,\n) -> xr.Dataset:\n    """Open one calendar-month analysis window."""\n    start, end = month_window(year, month)\n    return subset_era5_window(\n        ds,\n        start,\n        end,\n        domain=domain,\n        variables=variables,\n        level=level,\n    )\n\n\ndef open_baseline_window(\n    ds: xr.Dataset,\n    *,\n    domain: BoundingBox = WORKING_DOMAIN,\n    variables: Sequence[str] = ANALYSIS_VARIABLES,\n    level: int = 925,\n) -> xr.Dataset:\n    """Open the known February 2018 baseline event window."""\n    return subset_era5_window(\n        ds,\n        BASELINE_START_UTC,\n        BASELINE_END_UTC,\n        domain=domain,\n        variables=variables,\n        level=level,\n    )\n', 'historical_catalog': '"""Safe, resumable helpers for a full-record historical JPCZ catalog.\n\nThis module deliberately does not modify the existing 2000--2025 catalog.\nIt creates a second catalog whose anomaly baseline is formed from complete\nNDJF seasons, while allowing matching 2000--2025 monthly D12 checkpoints to\nbe reused as read-only inputs.\n"""\n\nfrom __future__ import annotations\n\nimport calendar\nimport gc\nfrom collections.abc import Iterable, Sequence\nfrom dataclasses import asdict, dataclass\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport xarray as xr\nimport metpy.calc as mpcalc\nfrom metpy.units import units\n\nfrom .detect import (\n    DetectionGeometry,\n    compute_rolling_mean,\n    count_events_from_threshold,\n    threshold_from_std,\n)\n\n\n@dataclass(frozen=True)\nclass NDJFMonth:\n    """One calendar month belonging to a complete NDJF season."""\n\n    season_year: int\n    year: int\n    month: int\n\n\ndef complete_ndjf_months(\n    first_season_year: int = 1940,\n    last_season_year: int = 2025,\n) -> tuple[NDJFMonth, ...]:\n    """Return Nov--Feb records for complete NDJF seasons.\n\n    ``season_year=1940`` means November--December 1940 followed by\n    January--February 1941.  This intentionally avoids the partial\n    January--February 1940 season, for which November--December 1939 are not\n    available in ERA5.\n    """\n    if last_season_year < first_season_year:\n        raise ValueError("last_season_year must be >= first_season_year")\n\n    records: list[NDJFMonth] = []\n    for season_year in range(first_season_year, last_season_year + 1):\n        records.extend(\n            [\n                NDJFMonth(season_year, season_year, 11),\n                NDJFMonth(season_year, season_year, 12),\n                NDJFMonth(season_year, season_year + 1, 1),\n                NDJFMonth(season_year, season_year + 1, 2),\n            ]\n        )\n    return tuple(records)\n\n\ndef checkpoint_path_for_month(directory: str | Path, record: NDJFMonth) -> Path:\n    """Return the standard D12 checkpoint filename for a calendar month."""\n    return Path(directory) / f"{record.year}_{record.month:02d}_D.nc"\n\n\ndef find_existing_checkpoint(\n    record: NDJFMonth,\n    *,\n    legacy_monthly_dir: str | Path,\n    legacy_december_dir: str | Path | None = None,\n) -> Path | None:\n    """Find a compatible old checkpoint without writing to its directory."""\n    monthly_path = checkpoint_path_for_month(legacy_monthly_dir, record)\n    if monthly_path.exists():\n        return monthly_path\n    if record.month == 12 and legacy_december_dir is not None:\n        december_path = Path(legacy_december_dir) / f"december_{record.year}_D.nc"\n        if december_path.exists():\n            return december_path\n    return None\n\n\ndef validate_d12_checkpoint(path: str | Path, record: NDJFMonth) -> tuple[bool, str]:\n    """Validate that a checkpoint has the expected month-long trailing-D12 series.\n\n    This checks the timestamp convention and the expected number of valid\n    values.  It cannot reconstruct the original polygon geometry from a\n    one-dimensional D12 file, so the workflow separately records that legacy\n    files are only reused when they came from the established matching ERA5\n    detector.\n    """\n    checkpoint = Path(path)\n    if not checkpoint.exists():\n        return False, "file missing"\n    try:\n        D = xr.open_dataarray(checkpoint)\n        if "time" not in D.dims:\n            return False, "missing time dimension"\n        timestamps = pd.DatetimeIndex(D.time.values)\n        expected_count = calendar.monthrange(record.year, record.month)[1] * 24 - 11\n        expected_start = pd.Timestamp(record.year, record.month, 1, 11)\n        expected_end = pd.Timestamp(\n            record.year,\n            record.month,\n            calendar.monthrange(record.year, record.month)[1],\n            23,\n        )\n        if len(timestamps) != expected_count:\n            return False, f"expected {expected_count} D12 values, found {len(timestamps)}"\n        if timestamps.empty or timestamps[0] != expected_start or timestamps[-1] != expected_end:\n            return False, "unexpected D12 time coverage"\n        if not timestamps.is_monotonic_increasing or timestamps.has_duplicates:\n            return False, "timestamps are not unique and increasing"\n        values = np.asarray(D.values)\n        if not np.isfinite(values).all():\n            return False, "contains non-finite D12 values"\n    except Exception as error:  # pragma: no cover - defensive Drive/file handling\n        return False, f"unreadable checkpoint: {type(error).__name__}: {error}"\n    finally:\n        try:\n            D.close()\n        except (NameError, AttributeError):\n            pass\n    return True, "valid"\n\n\ndef write_inventory(\n    records: Sequence[dict[str, object]], path: str | Path) -> pd.DataFrame:\n    """Atomically persist the Drive-resume inventory and return it."""\n    inventory_path = Path(path)\n    inventory_path.parent.mkdir(parents=True, exist_ok=True)\n    inventory = pd.DataFrame(records).sort_values(["season_year", "year", "month"])\n    temporary_path = inventory_path.with_suffix(inventory_path.suffix + ".tmp")\n    inventory.to_csv(temporary_path, index=False)\n    temporary_path.replace(inventory_path)\n    return inventory\n\n\ndef write_dataframe_atomic(frame: pd.DataFrame, path: str | Path, *, index: bool = False) -> Path:\n    """Write a CSV through a temporary sibling file, then atomically replace it."""\n    destination = Path(path)\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    temporary_path = destination.with_name(f".{destination.stem}.tmp{destination.suffix}")\n    frame.to_csv(temporary_path, index=index)\n    temporary_path.replace(destination)\n    return destination\n\n\ndef write_dataarray_atomic(array: xr.DataArray, path: str | Path) -> Path:\n    """Atomically persist a small NetCDF checkpoint to Drive."""\n    destination = Path(path)\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    temporary_path = destination.with_name(f".{destination.stem}.tmp{destination.suffix}")\n    array.to_netcdf(temporary_path)\n    temporary_path.replace(destination)\n    return destination\n\n\ndef initialize_historical_inventory(\n    records: Sequence[NDJFMonth],\n    *,\n    historical_monthly_dir: str | Path,\n    legacy_monthly_dir: str | Path,\n    legacy_december_dir: str | Path | None = None,\n) -> list[dict[str, object]]:\n    """Describe whether each month is reusable, already built, or pending.\n\n    A historical-run checkpoint always takes precedence.  Existing catalog\n    checkpoints are only *read*; this keeps the established catalog safe.\n    """\n    historical_dir = Path(historical_monthly_dir)\n    rows: list[dict[str, object]] = []\n    for record in records:\n        new_path = checkpoint_path_for_month(historical_dir, record)\n        legacy_path = find_existing_checkpoint(\n            record,\n            legacy_monthly_dir=legacy_monthly_dir,\n            legacy_december_dir=legacy_december_dir,\n        )\n        new_valid, new_validation = validate_d12_checkpoint(new_path, record)\n        legacy_valid = False\n        legacy_validation = "not checked"\n        if legacy_path is not None:\n            legacy_valid, legacy_validation = validate_d12_checkpoint(legacy_path, record)\n\n        if new_valid:\n            status, source, selected_path, validation = "complete", "historical_checkpoint", new_path, new_validation\n        elif legacy_valid:\n            status, source, selected_path, validation = "complete", "reused_legacy_checkpoint", legacy_path, legacy_validation\n        else:\n            reason = new_validation if new_path.exists() else legacy_validation\n            status, source, selected_path, validation = "pending", "new_era5_calculation", new_path, reason\n        rows.append(\n            {\n                **asdict(record),\n                "status": status,\n                "source": source,\n                "checkpoint_path": str(selected_path),\n                "checkpoint_validation": validation,\n            }\n        )\n    return rows\n\n\ndef compute_polygon_mean_divergence_series_chunked(\n    window_ds: xr.Dataset,\n    *,\n    geometry: DetectionGeometry,\n    chunk_hours: int = 24,\n    progress_every_hours: int = 24,\n    progress_label: str = "Monthly divergence",\n    hourly_checkpoint_path: str | Path | None = None,\n) -> tuple[xr.DataArray, xr.DataArray]:\n    """Calculate hourly polygon divergence using fast, bounded batches.\n\n    A short wind-data chunk is loaded from ARCO.  MetPy calculates all of the\n    chunk\'s hourly divergence fields in one vectorized call, and those fields\n    are immediately reduced to polygon-mean scalars.  Thus a full month-long\n    divergence stack is never retained in memory.  The individual hourly\n    calculations are the same three-point MetPy finite differences used by\n    the established detector; batching changes only execution speed.\n\n    When ``hourly_checkpoint_path`` is supplied, the scalar hourly series is\n    atomically saved after every progress interval.  A rerun validates and\n    resumes that partial series, so an interrupted month does not start from\n    hour zero.  The final trailing 12-hour mean remains computed only from\n    the complete hourly series.\n    """\n    if chunk_hours < 1:\n        raise ValueError("chunk_hours must be at least one hour")\n    if progress_every_hours < 1:\n        raise ValueError("progress_every_hours must be at least one hour")\n    if "time" not in window_ds.dims:\n        raise ValueError("window_ds must have a time dimension")\n\n    n_hours = int(window_ds.sizes["time"])\n    timestamps = np.asarray(window_ds.time.values)\n    hourly_values: list[float] = []\n    resume_index = 0\n    checkpoint = Path(hourly_checkpoint_path) if hourly_checkpoint_path is not None else None\n\n    if checkpoint is not None and checkpoint.exists():\n        cached = None\n        try:\n            cached = xr.open_dataarray(checkpoint).load()\n            cached_values = np.asarray(cached.values, dtype=float)\n            cached_times = np.asarray(cached.time.values)\n            if (\n                "time" in cached.dims\n                and 0 < len(cached_values) <= n_hours\n                and np.isfinite(cached_values).all()\n                and np.array_equal(cached_times, timestamps[: len(cached_values)])\n            ):\n                hourly_values = cached_values.tolist()\n                resume_index = len(hourly_values)\n                print(f"{progress_label}: resuming {resume_index}/{n_hours} saved hourly fields")\n            else:\n                print(f"{progress_label}: ignoring incompatible hourly partial checkpoint")\n        except Exception as error:  # pragma: no cover - defensive Drive recovery\n            print(f"{progress_label}: ignoring unreadable hourly partial checkpoint ({type(error).__name__})")\n        finally:\n            if cached is not None:\n                cached.close()\n\n    weight_values = np.asarray(geometry.weights.values, dtype=float)\n    weight_total = float(weight_values.sum())\n    if not np.isfinite(weight_total) or weight_total <= 0:\n        raise ValueError("Polygon weights must have a positive finite sum")\n\n    dx_values = np.asarray(geometry.dx.magnitude)\n    dy_values = np.asarray(geometry.dy.magnitude)\n    for start in range(resume_index, n_hours, chunk_hours):\n        stop = min(start + chunk_hours, n_hours)\n        chunk = window_ds.isel(time=slice(start, stop)).load()\n        n_chunk_hours = int(chunk.sizes["time"])\n        # MetPy accepts an extra leading time dimension when dx/dy are expanded\n        # to match it.  This is mathematically identical to calling divergence\n        # once per hour, but removes hundreds of Python/MetPy dispatches.\n        dx_chunk = np.broadcast_to(dx_values, (n_chunk_hours, *dx_values.shape)) * geometry.dx.units\n        dy_chunk = np.broadcast_to(dy_values, (n_chunk_hours, *dy_values.shape)) * geometry.dy.units\n        divergence = mpcalc.divergence(\n            chunk["u_component_of_wind"].values * units("m/s"),\n            chunk["v_component_of_wind"].values * units("m/s"),\n            dx=dx_chunk,\n            dy=dy_chunk,\n            x_dim=-1,\n            y_dim=-2,\n        ).magnitude\n        polygon_means = (divergence * weight_values[None, :, :]).sum(axis=(-2, -1)) / weight_total\n        hourly_values.extend(np.asarray(polygon_means, dtype=float).tolist())\n        if stop % progress_every_hours == 0 or stop == n_hours:\n            print(f"{progress_label}: {stop}/{n_hours} hourly fields complete")\n        if checkpoint is not None and (stop % progress_every_hours == 0 or stop == n_hours):\n            partial_hourly = xr.DataArray(\n                np.asarray(hourly_values),\n                coords={"time": timestamps[:stop]},\n                dims="time",\n                name="polygon_mean_divergence_925hpa",\n                attrs={"units": "s^-1", "checkpoint": "resumable partial hourly series"},\n            )\n            write_dataarray_atomic(partial_hourly, checkpoint)\n            print(f"{progress_label}: hourly partial checkpoint saved ({stop}/{n_hours})")\n            del partial_hourly\n        del chunk, divergence, polygon_means, dx_chunk, dy_chunk\n        gc.collect()\n\n    hourly = xr.DataArray(\n        np.asarray(hourly_values),\n        coords={"time": timestamps},\n        dims="time",\n        name="polygon_mean_divergence_925hpa",\n        attrs={"units": "s^-1", "display_units": "1e-5 s^-1"},\n    )\n    D_12h = compute_rolling_mean(hourly, window=12, min_periods=12, name="D_12h")\n    return hourly, D_12h\n\n\ndef build_events_from_monthly_checkpoints(\n    inventory: pd.DataFrame,\n    *,\n    required_months: Iterable[int] = (11, 12, 1, 2),\n) -> tuple[pd.DataFrame, pd.DataFrame]:\n    """Recalculate full-record thresholds and raw events from D12 files.\n\n    The inventory must be complete.  Thresholds are computed afresh from all\n    complete-season D12 values; no threshold from the established catalog is\n    reused.\n    """\n    required_months = tuple(required_months)\n    pending = inventory.loc[inventory["status"] != "complete"]\n    if not pending.empty:\n        raise RuntimeError(\n            f"Historical D12 inventory is incomplete: {len(pending)} month(s) pending."\n        )\n\n    records: list[dict[str, object]] = []\n    for row in inventory.sort_values(["season_year", "year", "month"]).itertuples(index=False):\n        path = Path(row.checkpoint_path)\n        if not path.exists():\n            raise FileNotFoundError(f"Inventory lists a missing D12 checkpoint: {path}")\n        D = xr.open_dataarray(path).load().dropna("time")\n        records.append(\n            {\n                "season_year": int(row.season_year),\n                "year": int(row.year),\n                "month": int(row.month),\n                "D": D,\n            }\n        )\n\n    threshold_rows: list[dict[str, float | int]] = []\n    thresholds: dict[int, float] = {}\n    for month in required_months:\n        D_month = xr.concat(\n            [record["D"] for record in records if record["month"] == month], dim="time"\n        ).sortby("time")\n        mean_value, std_value, threshold = threshold_from_std(D_month, n_std=2.0)\n        thresholds[month] = threshold\n        threshold_rows.append(\n            {\n                "month": month,\n                "D_mean_s-1": mean_value,\n                "D_std_s-1": std_value,\n                "threshold_s-1": threshold,\n                "threshold_1e5_s-1": threshold * 1e5,\n                "valid_D12_hours": int(D_month.sizes["time"]),\n            }\n        )\n\n    event_frames: list[pd.DataFrame] = []\n    for record in records:\n        events = count_events_from_threshold(record["D"], thresholds[record["month"]]).copy()\n        if events.empty:\n            continue\n        events["year"] = record["year"]\n        events["month"] = record["month"]\n        events["season_year"] = record["season_year"]\n        events["detection_threshold_s-1"] = thresholds[record["month"]]\n        events["detection_threshold_1e5_s-1"] = thresholds[record["month"]] * 1e5\n        event_frames.append(events)\n\n    events_df = (\n        pd.concat(event_frames, ignore_index=True).sort_values("event_peak").reset_index(drop=True)\n        if event_frames\n        else pd.DataFrame()\n    )\n    thresholds_df = pd.DataFrame(threshold_rows).sort_values("month").reset_index(drop=True)\n    return events_df, thresholds_df\n\n\ndef compare_merged_catalogs(\n    established_catalog: pd.DataFrame,\n    historical_catalog: pd.DataFrame,\n) -> tuple[pd.DataFrame, pd.DataFrame]:\n    """Match merged episodes by temporal overlap and report catalog changes.\n\n    An established episode is considered retained if any historical-baseline\n    merged episode overlaps its inclusive start--end interval.  This avoids\n    requiring identical peak hours when a changed threshold broadens or\n    shortens the same physical episode.\n    """\n    required = {"event_start", "event_end", "event_peak"}\n    for name, catalog in (("established", established_catalog), ("historical", historical_catalog)):\n        missing = required.difference(catalog.columns)\n        if missing:\n            raise ValueError(f"{name} catalog is missing required columns: {sorted(missing)}")\n\n    old = established_catalog.copy().reset_index(drop=True)\n    new = historical_catalog.copy().reset_index(drop=True)\n    for frame in (old, new):\n        for column in required:\n            try:\n                # Newer pandas defaults to one inferred format for the whole\n                # column; ``mixed`` keeps date-only and timestamp values safe.\n                frame[column] = pd.to_datetime(frame[column], format="mixed")\n            except (TypeError, ValueError):\n                # Retain compatibility with older pandas used by some Colab\n                # runtimes, where ``format=\'mixed\'`` is unavailable.\n                frame[column] = pd.to_datetime(frame[column])\n    old = old.sort_values("event_start").reset_index(drop=True)\n    new = new.sort_values("event_start").reset_index(drop=True)\n\n    rows: list[dict[str, object]] = []\n    old_match_count = np.zeros(len(old), dtype=int)\n    new_match_count = np.zeros(len(new), dtype=int)\n    for old_index, old_event in old.iterrows():\n        overlapping = new.loc[\n            (new["event_start"] <= old_event["event_end"])\n            & (new["event_end"] >= old_event["event_start"])\n        ]\n        for new_index, new_event in overlapping.iterrows():\n            overlap_start = max(old_event["event_start"], new_event["event_start"])\n            overlap_end = min(old_event["event_end"], new_event["event_end"])\n            overlap_hours = int((overlap_end - overlap_start).total_seconds() / 3600) + 1\n            rows.append(\n                {\n                    "comparison_status": "shared_temporal_overlap",\n                    "established_event_id": old_index + 1,\n                    "historical_event_id": new_index + 1,\n                    "established_start": old_event["event_start"],\n                    "established_end": old_event["event_end"],\n                    "established_peak": old_event["event_peak"],\n                    "historical_start": new_event["event_start"],\n                    "historical_end": new_event["event_end"],\n                    "historical_peak": new_event["event_peak"],\n                    "overlap_hours": overlap_hours,\n                    "peak_offset_hours": (new_event["event_peak"] - old_event["event_peak"]).total_seconds() / 3600,\n                }\n            )\n            old_match_count[old_index] += 1\n            new_match_count[new_index] += 1\n\n    for old_index, old_event in old.iterrows():\n        if old_match_count[old_index] == 0:\n            rows.append(\n                {\n                    "comparison_status": "established_only_not_flagged_historical",\n                    "established_event_id": old_index + 1,\n                    "historical_event_id": pd.NA,\n                    "established_start": old_event["event_start"],\n                    "established_end": old_event["event_end"],\n                    "established_peak": old_event["event_peak"],\n                    "historical_start": pd.NaT,\n                    "historical_end": pd.NaT,\n                    "historical_peak": pd.NaT,\n                    "overlap_hours": 0,\n                    "peak_offset_hours": np.nan,\n                }\n            )\n\n    for new_index, new_event in new.iterrows():\n        if new_match_count[new_index] == 0:\n            rows.append(\n                {\n                    "comparison_status": "historical_only_newly_flagged",\n                    "established_event_id": pd.NA,\n                    "historical_event_id": new_index + 1,\n                    "established_start": pd.NaT,\n                    "established_end": pd.NaT,\n                    "established_peak": pd.NaT,\n                    "historical_start": new_event["event_start"],\n                    "historical_end": new_event["event_end"],\n                    "historical_peak": new_event["event_peak"],\n                    "overlap_hours": 0,\n                    "peak_offset_hours": np.nan,\n                }\n            )\n\n    comparison = pd.DataFrame(rows)\n    summary = pd.DataFrame(\n        [\n            {"metric": "established_merged_episodes", "value": len(old)},\n            {"metric": "historical_merged_episodes", "value": len(new)},\n            {\n                "metric": "established_episodes_retained_by_temporal_overlap",\n                "value": int((old_match_count > 0).sum()),\n            },\n            {\n                "metric": "established_episodes_not_flagged_historical",\n                "value": int((old_match_count == 0).sum()),\n            },\n            {\n                "metric": "historical_only_newly_flagged_episodes",\n                "value": int((new_match_count == 0).sum()),\n            },\n        ]\n    )\n    return comparison, summary\n', 'seasonal_update': '"""Extend the established catalog without changing its anomaly baseline."""\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nfrom pathlib import Path\nimport shutil\n\nimport numpy as np\nimport pandas as pd\nimport xarray as xr\n\nfrom .config import JPCZ_POLYGON_VERTICES, ARCO_ERA5_ZARR_STORE\nfrom .detect import count_events_from_threshold, merge_nearby_events, prepare_detection_geometry, threshold_from_std\nfrom .era5 import subset_era5_window\nfrom .historical_catalog import (\n    NDJFMonth, compute_polygon_mean_divergence_series_chunked, find_existing_checkpoint,\n    validate_d12_checkpoint, write_dataframe_atomic, write_dataarray_atomic,\n)\n\nDATE_COLUMNS = (\'event_start\', \'event_end\', \'event_peak\')\nRAW_NAME = \'jpcz_catalog_ndjf.csv\'\nMERGED_NAME = \'jpcz_catalog_ndjf_merged_12h.csv\'\nDETECTOR = {\n    \'version\': \'established_shinoda_era5_frozen_monthly_v1\', \'pressure_hpa\': 925,\n    \'polygon\': JPCZ_POLYGON_VERTICES, \'weighting\': \'cos(latitude)\',\n    \'divergence\': \'MetPy divergence; lat_lon_grid_deltas; established Notebook 04\',\n    \'rolling_hours\': 12, \'rolling_label\': \'trailing window end\',\n    \'month_boundary\': \'11-hour warmup each calendar month, matching Notebook 04\',\n    \'criterion\': \'D12 < frozen month-specific baseline mean - 2 population std\',\n    \'merge_gap_hours\': 12, \'season_months\': [11, 12, 1, 2],\n}\n\n\ndef read_events(path):\n    frame = pd.read_csv(path)\n    for name in DATE_COLUMNS:\n        frame[name] = pd.to_datetime(frame[name], utc=True, format=\'mixed\').dt.tz_localize(None)\n    if frame[list(DATE_COLUMNS)].isna().any().any():\n        raise ValueError(f\'Missing event timestamps in {path}\')\n    if frame.duplicated(list(DATE_COLUMNS)).any():\n        raise ValueError(f\'Duplicate event identities in {path}\')\n    return frame\n\n\ndef event_keys(frame):\n    return frame[list(DATE_COLUMNS)].apply(lambda row: \'|\'.join(t.isoformat() for t in row), axis=1)\n\n\ndef inspect_catalog(root):\n    """Inspect actual Drive files; never infer the user\'s count from repository CSVs."""\n    root = Path(root)\n    raw, merged = read_events(root / RAW_NAME), read_events(root / MERGED_NAME)\n    missing = [r.event_start for r in raw.itertuples() if not\n               ((merged.event_start <= r.event_start) & (merged.event_end >= r.event_end)).any()]\n    if missing:\n        raise ValueError(\'Saved raw and merged catalogs disagree. Run the current Notebook 06 before extending.\')\n    expected = merge_nearby_events(raw, max_gap_hours=12)\n    if set(event_keys(expected)) != set(event_keys(merged)):\n        raise ValueError(\'The saved merged catalog does not match a 12-hour merge of the saved raw catalog. \'\n                         \'Review the Notebook 06 version/settings before extending.\')\n    last = raw.event_end.max()\n    if pd.isna(last):\n        raise ValueError(\'The established catalog is empty; build it with Notebook 04 first.\')\n    record = NDJFMonth(last.year, last.year, last.month)\n    path = find_existing_checkpoint(record, legacy_monthly_dir=root / \'ndjf_monthly\',\n                                    legacy_december_dir=root / \'december_yearly\')\n    valid, reason = validate_d12_checkpoint(path, record) if path else (False, \'not found\')\n    return {\'raw_events\': len(raw), \'merged_events\': len(merged),\n            \'first_peak\': str(merged.event_peak.min()), \'last_peak\': str(merged.event_peak.max()),\n            \'last_event_month_complete\': valid, \'checkpoint_note\': reason,\n            \'suggested_update_start\': str((last.to_period(\'M\') + 1).start_time.date()) if valid else None,\n            \'detector\': DETECTOR}\n\n\ndef frozen_thresholds(root):\n    """Use Notebook 04 thresholds, or reconstruct them from its complete saved months."""\n    root = Path(root)\n    path = root / \'ndjf_monthly_thresholds.csv\'\n    if path.exists():\n        frame = pd.read_csv(path)\n        source = str(path)\n    else:\n        # Only years in the saved established raw catalog, never the new winter.\n        raw = read_events(root / RAW_NAME)\n        years = sorted(raw.event_start.dt.year.unique())\n        rows = []\n        for month in (11, 12, 1, 2):\n            series = []\n            for year in years:\n                record = NDJFMonth(int(year), int(year), month)\n                saved = find_existing_checkpoint(record, legacy_monthly_dir=root / \'ndjf_monthly\',\n                                                  legacy_december_dir=root / \'december_yearly\')\n                valid, _ = validate_d12_checkpoint(saved, record) if saved else (False, \'\')\n                if not valid:\n                    raise FileNotFoundError(\n                        f\'Missing complete baseline checkpoint {year}-{month:02d}. Restore \'\n                        \'ndjf_monthly_thresholds.csv from the catalog run; do not fit thresholds to this winter.\')\n                with xr.open_dataarray(saved) as array:\n                    series.append(array.load())\n            mean, std, value = threshold_from_std(xr.concat(series, dim=\'time\'))\n            rows.append({\'month\': month, \'D_mean_s-1\': mean, \'D_std_s-1\': std, \'threshold_s-1\': value})\n        frame, source = pd.DataFrame(rows), \'reconstructed from complete established-catalog years: \' + str(years)\n    if frame.month.duplicated().any() or set(frame.month) != {11, 12, 1, 2}:\n        raise ValueError(\'Need exactly one threshold for each NDJF calendar month.\')\n    if not np.isfinite(frame[\'threshold_s-1\']).all():\n        raise ValueError(\'Thresholds must be finite.\')\n    raw = read_events(root / RAW_NAME)\n    if \'detection_threshold_s-1\' in raw:\n        recorded = pd.to_numeric(raw[\'detection_threshold_s-1\'], errors=\'raise\')\n        expected = raw.event_start.dt.month.map(frame.set_index(\'month\')[\'threshold_s-1\'])\n        if not np.isfinite(recorded).all() or not np.isclose(recorded, expected, rtol=1e-10, atol=1e-14).all():\n            raise ValueError(\'Threshold file disagrees with thresholds recorded in the established raw catalog. \'\n                             \'Restore the matching Notebook 04 threshold table before extending.\')\n    return frame, source\n\n\ndef initialize_update(root, update_start=None):\n    """Create an immutable baseline backup once; subsequent runs use that same baseline."""\n    root = Path(root)\n    folder = root / \'seasonal_updates\'\n    manifest_path = folder / \'baseline.json\'\n    if manifest_path.exists():\n        manifest = json.loads(manifest_path.read_text())\n        if update_start and pd.Timestamp(update_start) != pd.Timestamp(manifest[\'update_start\']):\n            raise ValueError(\'Keep the saved update_start when extending this same rolling catalog.\')\n        # Detect accidental edits to the frozen threshold record.\n        payload = (folder / \'frozen_thresholds.csv\').read_bytes()\n        if hashlib.sha256(payload).hexdigest() != manifest[\'threshold_sha256\']:\n            raise ValueError(\'Frozen thresholds have changed; restore the original baseline copy.\')\n        if hashlib.sha256((folder / (\'baseline_\' + RAW_NAME)).read_bytes()).hexdigest() != manifest[\'baseline_raw_sha256\']:\n            raise ValueError(\'Frozen baseline raw catalog has changed; restore its original copy.\')\n        return manifest\n    audit = inspect_catalog(root)\n    update_start = update_start or audit[\'suggested_update_start\']\n    if not update_start:\n        raise ValueError(\'Cannot infer catalog coverage from the last event alone. Enter UPDATE_START_UTC \'\n                         \'after verifying the end of your completed Notebook 04 run.\')\n    start = pd.Timestamp(update_start)\n    raw = read_events(root / RAW_NAME)\n    if start != start.to_period(\'M\').start_time or start <= raw.event_end.max():\n        raise ValueError(\'Start must be a calendar-month boundary after all established raw events.\')\n    thresholds, source = frozen_thresholds(root)\n    folder.mkdir(parents=True, exist_ok=True)\n    for name in (RAW_NAME, MERGED_NAME):\n        backup = folder / (\'baseline_\' + name)\n        if not backup.exists():\n            shutil.copy2(root / name, backup)\n    write_dataframe_atomic(thresholds, folder / \'frozen_thresholds.csv\')\n    manifest = {\'detector\': DETECTOR, \'update_start\': start.isoformat(), \'threshold_source\': source,\n                \'threshold_sha256\': hashlib.sha256((folder / \'frozen_thresholds.csv\').read_bytes()).hexdigest(),\n                \'baseline_raw_sha256\': hashlib.sha256((folder / (\'baseline_\' + RAW_NAME)).read_bytes()).hexdigest(),\n                \'baseline_merged_events\': audit[\'merged_events\'], \'era5_store\': ARCO_ERA5_ZARR_STORE}\n    manifest_path.write_text(json.dumps(manifest, indent=2) + \'\\n\')\n    return manifest\n\n\ndef availability(ds, allow_era5t=False):\n    """ARCO coordinates may extend past populated data; trust published inclusive stops."""\n    stable = ds.attrs.get(\'valid_time_stop\')\n    preliminary = ds.attrs.get(\'valid_time_stop_era5t\') if allow_era5t else None\n    stops = [pd.to_datetime(s, utc=True).tz_localize(None) for s in (stable, preliminary) if s]\n    if not stops:\n        raise ValueError(\'Archive has no valid_time_stop metadata; refusing to read unpopulated times.\')\n    return max(stops).floor(\'h\')\n\n\ndef update_catalog(root, ds, *, update_start=None, requested_end=None, allow_era5t=True, progress=print):\n    """Scan only the extension, checkpoint monthly scalars, then refresh the shared CSVs.\n\n    requested_end is inclusive. A pending tail is reported, not filled or counted as no events.\n    Preliminary months are re-read on each run and once more when finalized.\n    """\n    root = Path(root)\n    manifest = initialize_update(root, update_start)\n    folder = root / \'seasonal_updates\'\n    start = pd.Timestamp(manifest[\'update_start\'])\n    available = availability(ds, allow_era5t)\n    requested = pd.to_datetime(requested_end or pd.Timestamp.now(tz=\'UTC\').floor(\'h\'), utc=True).tz_localize(None)\n    stop = min(requested, available)\n    stable_stop = pd.to_datetime(ds.attrs[\'valid_time_stop\'], utc=True).tz_localize(None)\n    thresholds = pd.read_csv(folder / \'frozen_thresholds.csv\').set_index(\'month\')[\'threshold_s-1\']\n    baseline = read_events(folder / (\'baseline_\' + RAW_NAME))\n    # Do not erase later extensions if somebody requests an earlier end date on a rerun.\n    previous = folder / \'scan_manifest.json\'\n    if previous.exists() and stop < pd.Timestamp(json.loads(previous.read_text())[\'scanned_through\']):\n        raise ValueError(\'Requested/available end precedes an existing scan. Keep or extend the saved end date.\')\n    if stop < start:\n        return {\'status\': \'pending_era5\', \'available_through\': str(available), \'requested_through\': str(requested),\n                \'pending_after\': str(available), \'merged_events\': len(read_events(root / MERGED_NAME))}\n    event_frames, inventory = [], []\n    for period in pd.period_range(start, stop, freq=\'M\'):\n        if period.month not in thresholds.index:\n            continue\n        begin = period.start_time\n        end = min(period.end_time.floor(\'h\'), stop)\n        if end < begin + pd.Timedelta(hours=11):\n            continue\n        product = \'ERA5\' if end <= stable_stop else \'ERA5T\'\n        stamp = str(period)\n        path = folder / \'monthly\' / f\'{stamp}_D12.nc\'\n        cached = None\n        if path.exists():\n            with xr.open_dataarray(path) as saved:\n                if (saved.attrs.get(\'source_product\') == \'ERA5\' and product == \'ERA5\'\n                        and saved.attrs.get(\'scanned_through\') == end.isoformat()):\n                    cached = saved.load()\n        if cached is None:\n            progress(f\'{stamp}: {begin} to {end} ({product})\')\n            winds = subset_era5_window(ds, str(begin), str(end), variables=(\'u_component_of_wind\', \'v_component_of_wind\'))\n            expected = pd.date_range(begin, end, freq=\'h\')\n            if not pd.DatetimeIndex(winds.time.values).equals(expected):\n                raise ValueError(f\'{stamp}: missing hourly winds; catalog was not updated.\')\n            geometry = prepare_detection_geometry(winds.longitude, winds.latitude, JPCZ_POLYGON_VERTICES)\n            # Final data resume; preliminary scalar values must be refreshed when the archive is revised.\n            partial = folder / \'hourly\' / f\'{stamp}.nc\' if product == \'ERA5\' else None\n            hourly, d12 = compute_polygon_mean_divergence_series_chunked(\n                winds, geometry=geometry, chunk_hours=6, progress_every_hours=24,\n                progress_label=stamp, hourly_checkpoint_path=partial)\n            if not np.isfinite(hourly.values).all():\n                raise ValueError(f\'{stamp}: nonfinite winds/divergence; catalog was not updated.\')\n            cached = d12.dropna(\'time\')\n            cached.attrs.update(source_product=product, scanned_through=end.isoformat())\n            write_dataarray_atomic(cached, path)\n        events = count_events_from_threshold(cached, float(thresholds[period.month]))\n        if not events.empty:\n            events[\'year\'], events[\'month\'] = period.year, period.month\n            # Existing notebook labels winters by their Jan/Feb year.\n            events[\'season_year\'] = period.year + (period.month in (11, 12))\n            events[\'detection_threshold_s-1\'] = float(thresholds[period.month])\n            events[\'detection_threshold_1e5_s-1\'] = float(thresholds[period.month]) * 1e5\n            events[\'reference_product\'] = product\n            events[\'catalog_provisional\'] = (events.event_end + pd.Timedelta(hours=12) >= stop) | (product == \'ERA5T\')\n            event_frames.append(events)\n        inventory.append({\'month\': stamp, \'scanned_through\': str(end), \'source_product\': product,\n                          \'events\': len(events), \'checkpoint\': str(path)})\n    extension = pd.concat(event_frames, ignore_index=True) if event_frames else baseline.iloc[:0].copy()\n    combined = pd.concat([baseline, extension], ignore_index=True).sort_values(\'event_start\').reset_index(drop=True)\n    merged = merge_nearby_events(combined, max_gap_hours=12)\n    # Keep all existing display numbers on exact identities; assign new numbers only to changed/new events.\n    old = read_events(root / MERGED_NAME)\n    old_keys = event_keys(old)\n    numbers = pd.to_numeric(old.get(\'event_number\', pd.Series(np.arange(1, len(old) + 1))), errors=\'coerce\')\n    if numbers.isna().any() or not numbers.is_unique or (numbers <= 0).any():\n        numbers = pd.Series(np.arange(1, len(old) + 1))\n    lookup = dict(zip(old_keys, numbers.astype(int)))\n    next_number = int(numbers.max()) + 1\n    allocated = []\n    for key in event_keys(merged):\n        if key not in lookup:\n            lookup[key] = next_number\n            next_number += 1\n        allocated.append(lookup[key])\n    merged[\'event_number\'] = allocated\n    merged[\'review_key\'] = event_keys(merged)\n    # Propagate a provisional fragment to its entire merged episode.\n    extension_provisional = extension.loc[extension.get(\'catalog_provisional\', pd.Series(False, index=extension.index)).eq(True)]\n    merged[\'catalog_provisional\'] = [bool(((extension_provisional.event_start <= row.event_end) &\n                                          (extension_provisional.event_end >= row.event_start)).any())\n                                     for row in merged.itertuples()]\n    changes = merged.copy()\n    changes[\'update_status\'] = np.where(changes.review_key.isin(old_keys), \'unchanged\', \'new_or_revised\')\n    # Stage both files and a ledger before promotion. The immutable baseline remains available for recovery.\n    write_dataframe_atomic(combined, folder / \'candidate_raw.csv\')\n    write_dataframe_atomic(merged, folder / \'candidate_merged.csv\')\n    write_dataframe_atomic(changes, folder / \'event_changes.csv\')\n    write_dataframe_atomic(pd.DataFrame(inventory), folder / \'monthly_scan_status.csv\')\n    write_dataframe_atomic(combined, root / RAW_NAME)\n    write_dataframe_atomic(merged, root / MERGED_NAME)\n    result = {\'status\': \'updated\' if requested <= available else \'updated_with_pending_era5_tail\',\n              \'scanned_through\': stop.isoformat(), \'available_through\': available.isoformat(),\n              \'requested_through\': requested.isoformat(), \'merged_events\': len(merged),\n              \'new_or_revised\': int(changes.update_status.ne(\'unchanged\').sum()),\n              \'provisional_events\': int(merged.catalog_provisional.sum()), \'allow_era5t\': allow_era5t,\n              \'detector\': manifest[\'detector\']}\n    previous.write_text(json.dumps(result, indent=2) + \'\\n\')\n    progress(str(result))\n    return result\n'}
package_name = 'jpcz_update_runtime'
package = types.ModuleType(package_name)
package.__path__ = []
sys.modules[package_name] = package
for name, source in sources.items():
    qualified = package_name + '.' + name
    module = types.ModuleType(qualified)
    module.__package__ = package_name
    sys.modules[qualified] = module
    exec(compile(source, qualified, 'exec'), module.__dict__)
from jpcz_update_runtime.seasonal_update import inspect_catalog, initialize_update, update_catalog
from jpcz_update_runtime.era5 import open_arco_era5
print('Established hourly MetPy / trailing-D12 / frozen monthly thresholds / 12-hour merge loaded.')


## Inspect before scanning

The preview reads only your Drive inventory. It prints the actual count, latest event,
and whether the last event's month has a complete detector checkpoint. The last event alone
cannot prove data coverage. If inference is unavailable, enter the first unscanned calendar
month in `UPDATE_START_UTC` after checking your saved Notebook 04 coverage.

Keep the saved start date on subsequent runs; extend only the end. If the threshold CSV is
missing, recovery requires every complete saved NDJF month in the established catalog years.
If those files are absent, restore `ndjf_monthly_thresholds.csv` from the original run.


In [ ]:
# @title 4 · Inspect the established inventory and archive availability
import json
audit = inspect_catalog(DRIVE_ROOT)
print(json.dumps(audit, indent=2))
era5 = open_arco_era5(chunks=None)
print({key: era5.attrs.get(key) for key in ('valid_time_start', 'valid_time_stop',
                                          'valid_time_stop_era5t', 'last_updated')})


## Scan and save new events

Run this cell to update `jpcz_catalog_ndjf.csv` and `jpcz_catalog_ndjf_merged_12h.csv`,
the same filenames used by 35–37. The immutable original copies and frozen detector settings
are in `seasonal_updates/`. `event_changes.csv` identifies new/revised episodes.
Existing event numbers survive exact start/end/peak matches. A changed episode gets a new
number and needs a new review. Unchecked events remain an **Unknown** verification stratum.
New rows do not inherit the old monsoon or higher-vorticity classifications.

An event still touching the scan boundary, or one detected from ERA5T, is marked provisional.
Its end/peak may change on a later scan. The next run reconstructs the extension and merges
it with the frozen baseline, rather than appending duplicate rows.

ARCO's published data stop controls what can be read. Coordinates alone do not prove fields
are populated. An event on November 19 will wait for reference data; missing days are reported
as pending. [ARCO documentation](https://github.com/google-research/arco-era5#overview) describes
roughly one week of ERA5T delay and about three months for final ERA5. Preliminary months
are reread on each run, and again when final data replace them.


In [ ]:
# @title 5 · Run the seasonal update and save the shared catalog
manifest = initialize_update(DRIVE_ROOT, UPDATE_START_UTC or None)
print('Frozen update start:', manifest['update_start'])
print('Threshold source:', manifest['threshold_source'])
result = update_catalog(DRIVE_ROOT, era5, update_start=UPDATE_START_UTC or None,
                        requested_end=UPDATE_THROUGH_UTC or None, allow_era5t=ALLOW_ERA5T)
print(json.dumps(result, indent=2))
status_path = DRIVE_ROOT / 'seasonal_updates/monthly_scan_status.csv'
if status_path.exists():
    display(pd.read_csv(status_path))


In [ ]:
# @title 6 · Search updated events by UTC date
SEARCH_START_UTC = ''  # Example: '2026-11-19'
SEARCH_END_UTC = ''  # Exclusive UTC boundary; e.g. '2026-11-20' for the 19th only.
from jpcz_update_runtime.seasonal_update import read_events
current = read_events(DRIVE_ROOT / 'jpcz_catalog_ndjf_merged_12h.csv')
found = current.copy()
if SEARCH_START_UTC:
    found = found.loc[found.event_end.ge(pd.Timestamp(SEARCH_START_UTC))]
if SEARCH_END_UTC:
    found = found.loc[found.event_start.lt(pd.Timestamp(SEARCH_END_UTC))]
print(f'{len(current)} total events; {len(found)} match the date search.')
columns = ['event_number', 'event_start', 'event_peak', 'event_end',
           'event_peak_D_1e5_s-1', 'catalog_provisional']
display(found[[column for column in columns if column in found]])
changes = DRIVE_ROOT / 'seasonal_updates/event_changes.csv'
if changes.exists():
    updates = pd.read_csv(changes)
    display(updates.loc[updates.update_status.ne('unchanged')])


## Carry the new cases into the figures

1. **Notebook 35:** rerun the catalog cell or click **Refresh catalog**. Screen new cases for
   vortex association; missing/failed screens stay Unknown in Notebook 37's strata. IMERG
   Final and satellite data have their own availability; they do not gate ensemble verification.
2. **Notebook 36:** rerun the inventory cell, then recreate the browser. Select a new event and
   check data. Exact 1-, 5-, and 10-day runs are requested relative to its onset target.
3. **Notebook 37:** choose **Catalog**, leave the end date open, reload the inventory, and run
   verification. Completed WN2 checkpoints are reused, missing runs are retried, ERA5T is
   refreshed, and all ranks/figures are regenerated from the current catalog. Use the regular
   winter schedule for broader convergence reliability; it must also include non-events.

If WN2 does not contain a new date, it stays missing in the status table until that exact
WN2 initialization/lead becomes available. An optional explicit WN2 store override is supported.
A different WeatherNext model is not substituted. Updating CSVs does not run Colab notebooks
in the background: rerun these stages when you want new figures.
